# SynBioHub

![SynBioHub](https://proto-bio.github.io/proto-assets/images/tool/synbiohub/hero.png)

This notebook demonstrates `run_synbiohub_fetch`, which downloads one design from a SynBioHub registry as SBOL and returns it as typed parts and features. The SBOL is kept verbatim, and the output exports to SBOL, FASTA, or GenBank. The running example is the iGEM device `BBa_I13504`, a GFP expression cassette built from an RBS, the GFP coding sequence, and a double terminator.

The public SynBioHub instance requires a login for downloads, including public records. Set `SYNBIOHUB_EMAIL` and `SYNBIOHUB_PASSWORD` to your SynBioHub account before running.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("synbiohub")
display_overview("synbiohub")
display_docs_section("synbiohub", "Background")

## Available tools

In [ ]:
display_available_tools("synbiohub")

### `run_synbiohub_fetch`

`run_synbiohub_fetch` takes one design's identifier: a SynBioHub link, an iGEM Registry part link, or an iGEM part name such as `BBa_I13504`. It confirms the object is a ComponentDefinition, downloads its recursive SBOL, and parses it. `design` is the requested part and `subparts` holds every part it references. A SynBioHub link must be on the same host as `registry_url`, so an identifier cannot redirect retrieval to another server. Each fetch logs in with `SYNBIOHUB_EMAIL` and `SYNBIOHUB_PASSWORD` and logs out afterward; neither appears in cache keys or outputs.

In [ ]:
import os

from proto_tools import (
    SynBioHubFetchConfig,
    SynBioHubFetchInput,
    run_synbiohub_fetch,
)

assert os.environ.get("SYNBIOHUB_EMAIL") and os.environ.get("SYNBIOHUB_PASSWORD"), "set SYNBIOHUB_EMAIL and SYNBIOHUB_PASSWORD"

In [ ]:
# Display docs
display_api_reference("synbiohub", "input", "run_synbiohub_fetch")

# One design on the public registry
inputs = SynBioHubFetchInput(identifier="https://api.synbiohub.org/public/igem/BBa_I13504/1")

In [ ]:
# Display config docs
display_api_reference("synbiohub", "config", "run_synbiohub_fetch")

# Defaults use the public registry and read SYNBIOHUB_EMAIL and SYNBIOHUB_PASSWORD | see docs above for all fields
config = SynBioHubFetchConfig()

In [ ]:
# Run the fetch
result = run_synbiohub_fetch(inputs, config)

In [ ]:
# Display output docs
display_api_reference("synbiohub", "output", "run_synbiohub_fetch")

design = result.design
print(f"{design.display_id}: {design.description} ({design.molecule}, {len(design.sequence)} nt)")
print(f"Roles: {design.roles}")
print(f"{len(design.features)} features, {len(result.subparts)} subparts, {len(result.sbol):,} chars of SBOL")

#### Read the Composition

Each feature places a subpart on the design sequence with 1-indexed, inclusive coordinates. `subpart_uri` points into `subparts`, which carries each part's own description, roles, and sequence.

In [ ]:
subparts = {part.uri: part for part in result.subparts}

for feature in sorted(design.features, key=lambda f: f.locations[0].start):
    location = feature.locations[0]
    part = subparts[feature.subpart_uri]
    print(f"{location.start:>4}-{location.end:<4} ({location.strand})  {part.display_id:<10} {feature.roles}  {part.description}")

## Export Results

The output exports to JSON (the default), the verbatim SBOL, FASTA, or GenBank. FASTA and GenBank are views of the requested design; GenBank feature keys come from the Sequence Ontology roles.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

for file_format, suffix in [("json", "json"), ("sbol", "sbol"), ("fasta", "fasta"), ("genbank", "gb")]:
    result.export("synbiohub_bba_i13504", export_path=output_dir, file_format=file_format)
    print(f"Exported to {output_dir / f'synbiohub_bba_i13504.{suffix}'}")

print()
print((output_dir / "synbiohub_bba_i13504.gb").read_text().split("ORIGIN")[0])